# SV2 Analyzer
## For Parsing, interpreting and analyzing Verilog/System Verilog Testbenches

### Getting the RAG database files' paths

In [2]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/sysverilogstd/sysverilogstd.pdf
/kaggle/input/verilogstd/verilogstd.pdf
/kaggle/input/uvmstd/uvmstd.pdf


### Logging into Hugging face

In [2]:
from huggingface_hub import login
login()

### Installing Dependencies

In [3]:
!pip install faiss-cpu langchain langchain-community langchain-core pypdf sentence-transformers  
!pip install sentence-transformers PyPDF2 faiss-cpu
!pip install safetensors
!pip -q install "transformers>=4.43" "accelerate>=0.33" "bitsandbytes>=0.43" "autoawq>=0.2.7" torch --extra-index-url https://download.pytorch.org/whl/cu121

INFO: pip is looking at multiple versions of langchain-community to determine which version is compatible with other requirements. This could take a while.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 97.1 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 77.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 104.7 MB/s eta 0:00:00
  Attempting uninstall: SQLAlchemy
    Found existing installation: SQLAlchemy 1.2.19
    Uninstalling SQLAlchemy-1.2.19:
      Successfully uninstalled SQLAlchemy-1.2.19
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
xmanager 0.7.1 requires sqlalchemy==1.2.19, but you have sqlalchemy 2.0.46 which is incompatible.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 232.6/232.6 kB 5.9 MB/s eta 0:00:00a 0:00:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 74.3/74.3 kB 2.9 MB/

### General Imports

In [4]:
from langchain.vectorstores import FAISS
from langchain.embeddings import HuggingFaceEmbeddings
from langchain.document_loaders import PyPDFLoader
from langchain.text_splitter import CharacterTextSplitter
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from langchain.output_parsers import StructuredOutputParser, ResponseSchema
from langchain.prompts import PromptTemplate
import re
from transformers import AutoModel
import numpy as np
from sentence_transformers import SentenceTransformer
from PyPDF2 import PdfReader
import faiss
from pathlib import Path

2026-01-26 14:54:34.036525: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1769439274.215016      55 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1769439274.267861      55 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1769439274.676159      55 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1769439274.676204      55 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1769439274.676208      55 computation_placer.cc:177] computation placer alr

### Reading Verilog, System Verilog and UVM standards,
### Then creating the vector database for them using the langchain functions
#### The list has three databases one for each standard for speeding up the search process

In [5]:
# 1️⃣ List of PDF paths
pdf_paths = [
    "/kaggle/input/verilogstd/verilogstd.pdf",
    "/kaggle/input/sysverilogstd/sysverilogstd.pdf",
    "/kaggle/input/uvmstd/uvmstd.pdf"
]
embedding_model_name = "BAAI/bge-m3"
embedding = HuggingFaceEmbeddings(model_name=embedding_model_name)
vectordb = []
for path in pdf_paths:
    loader = PyPDFLoader(path)
    documents = loader.load()
    text_splitter = CharacterTextSplitter(chunk_size=1000, chunk_overlap=50)
    chunks = text_splitter.split_documents(documents)
    temp_db = FAISS.from_documents(chunks, embedding)
    vectordb.append(temp_db) 
    

/tmp/ipykernel_55/495519917.py:8: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the :class:`~langchain-huggingface package and should be used instead. To use it run `pip install -U :class:`~langchain-huggingface` and import as `from :class:`~langchain_huggingface import HuggingFaceEmbeddings``.
  embedding = HuggingFaceEmbeddings(model_name=embedding_model_name)


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/123 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/54.0 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/444 [00:00<?, ?B/s]

sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/17.1M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/964 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/191 [00:00<?, ?B/s]

In [7]:
torch.cuda.empty_cache()
import gc
gc.collect()

0

### Installing the main model for the project as a 4-bit quantized model

In [8]:
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
import torch

# Configure 4-bit quantization
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4"
)

# Load model with quantization
model = AutoModelForCausalLM.from_pretrained(
    "Qwen/Qwen2.5-14B-Instruct",
    quantization_config=bnb_config,
    device_map="auto",
    trust_remote_code=True
)

model.eval()

# Load tokenizer — KEY CHANGE BELOW
tokenizer = AutoTokenizer.from_pretrained(
    "Qwen/Qwen2.5-14B-Instruct",
    trust_remote_code=True,
    model_max_length=131072
)



config.json:   0%|          | 0.00/663 [00:00<?, ?B/s]

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 8 files:   0%|          | 0/8 [00:00<?, ?it/s]

model-00007-of-00008.safetensors:   0%|          | 0.00/4.00G [00:00<?, ?B/s]

model-00005-of-00008.safetensors:   0%|          | 0.00/3.98G [00:00<?, ?B/s]

...

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

In [10]:
torch.cuda.empty_cache()
import gc
gc.collect()

0

### Installing the coder model, its role is to parse the input code or summarize the LLM response depending on the user query

In [11]:
from transformers import AutoTokenizer, AutoModelForCausalLM
tokenizer_summary = AutoTokenizer.from_pretrained("deepseek-ai/deepseek-coder-1.3b-instruct", trust_remote_code=True)
model_summary = AutoModelForCausalLM.from_pretrained("deepseek-ai/deepseek-coder-1.3b-instruct", trust_remote_code=True, torch_dtype=torch.bfloat16).cuda()
model_summary.eval()



tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

config.json:   0%|          | 0.00/631 [00:00<?, ?B/s]

`torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors:   0%|          | 0.00/2.69G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/119 [00:00<?, ?B/s]

LlamaForCausalLM(
  (model): LlamaModel(
    (embed_tokens): Embedding(32256, 2048)
    (layers): ModuleList(
      (0-23): 24 x LlamaDecoderLayer(
        (self_attn): LlamaAttention(
          (q_proj): Linear(in_features=2048, out_features=2048, bias=False)
          (k_proj): Linear(in_features=2048, out_features=2048, bias=False)
          (v_proj): Linear(in_features=2048, out_features=2048, bias=False)
          (o_proj): Linear(in_features=2048, out_features=2048, bias=False)
        )
        (mlp): LlamaMLP(
          (gate_proj): Linear(in_features=2048, out_features=5504, bias=False)
          (up_proj): Linear(in_features=2048, out_features=5504, bias=False)
          (down_proj): Linear(in_features=5504, out_features=2048, bias=False)
          (act_fn): SiLUActivation()
        )
        (input_layernorm): LlamaRMSNorm((2048,), eps=1e-06)
        (post_attention_layernorm): LlamaRMSNorm((2048,), eps=1e-06)
      )
    )
    (norm): LlamaRMSNorm((2048,), eps=1e-06)
    (r

In [13]:
torch.cuda.empty_cache()
import gc
gc.collect()

0

### Preparing the parsing response schema and telling the parser model how to respond

In [14]:
signals_schema = ResponseSchema(
    name="Signals",
    description="Defined signals in the testbench"
)
clock_schema = ResponseSchema(
    name="Clocks",
    description="Defined clocks in the testbench Exclude reset signals like rst, rst_n."
)
dut_schema = ResponseSchema(
    name="DUT",
    description="Names of instantiated design modules in the testbench e.g. adder, riscv etc."
)
block_schema = ResponseSchema(
    name="Blocks",
    description="Type and count each type of behavioural blocks in the testbench e.g. always, initial etc."
)

response_schemas = [signals_schema, clock_schema, dut_schema, block_schema]
output_parser = StructuredOutputParser.from_response_schemas(response_schemas)
format_instructions = output_parser.get_format_instructions()

testbench_extraction_template = """
You are a smart digital verification assistant that parses verilog, system verilog and uvm testbenches accurately.

Extract the defined signals with their data types in the testbench, the defined clocks in the testbench, the instantiated modules in the testbench and the type of used behavioural blocks in the testbench.

Respond ONLY in JSON format as follows:
{format_instructions}

RULES for your output:
1. NEVER explain, apologize, or add commentary.
2. NEVER say "it's not possible" or "simplified" or "I can't".
3. You are a machine parser. Output ONLY valid JSON with this exact schema:
  "Signals": ["signal1", "signal2", ...],   // ARRAY of signal NAMES only consider all types logic, reg, wire, int, ...(no newlines)
  "Clocks": ["clk", "sys_clk", ...],       // ARRAY of clock signal names 
  "DUT": ["dut" , "uut" , ...]             // ARRAY Of names of DUT instances
  "Blocks": ["always", "initial", ...]     // ARRAY of block types
4. If a field has no items, use an empty list [] or 0 for DUT.
5. DO NOT include markdown, code fences, or extra text or give python codes.
6. DO NOT assume non-present signals or blocks give only those in the following input
7. signals including clk or clock in their names are clocks 




Now extract from the following input:
"{user_input}"
"""

### Function for bug fix of the parsing

In [15]:
def remove_reset_clocks(json_obj):
    if "Clocks" not in json_obj or not isinstance(json_obj["Clocks"], list):
        return json_obj

    json_obj["Clocks"] = [
        clk for clk in json_obj["Clocks"]
        if not any(keyword in str(clk).lower() for keyword in ("rst", "reset"))
    ]

    return json_obj

### Function for sending the query to be parsed 

In [19]:
def parse_query(user_input , format_instructions):
    prompt = PromptTemplate(
    template=testbench_extraction_template,
        input_variables=["user_input", "format_instructions"]
        ).format(user_input=user_input, format_instructions=format_instructions)
    messages=[
        { 'role' : 'system' , 'content' : "you are proficient digital verification analyzer assistant. Your duty is to analyze and parse the given testbench code carefully line by line word by word."},
        { 'role': 'user', 'content': prompt}
    ]
    inputs = tokenizer_summary.apply_chat_template(messages, add_generation_prompt=True, return_tensors="pt").to(model_summary.device)
    # tokenizer.eos_token_id is the id of <|EOT|> token
    outputs = model_summary.generate(inputs, max_new_tokens=700, do_sample=True, num_return_sequences=1, eos_token_id=tokenizer_summary.eos_token_id)
    resp = tokenizer_summary.decode(outputs[0][len(inputs[0]):], skip_special_tokens=True)
    resp_json =  output_parser.parse(resp)
    resp_clean = remove_reset_clocks(resp_json)
    return resp_clean

### Function for interpreting the user code
#### Also, it is accompanied function that removes markdown format for getting clean text

In [23]:
# Cleaning Markdown text
def markdown_to_plain_text(md_text):
    text = md_text

    # 🔥 FIX: turn literal "\n" into real newlines
    text = text.replace("\\n", "\n")

    # Remove fenced code blocks (```lang ... ```)
    text = re.sub(r"```[\w+-]*\n[\s\S]*?```", "", text)

    # Remove inline code backticks
    text = re.sub(r"`([^`]*)`", r"\1", text)

    # Remove bold / italics markers
    text = re.sub(r"(\*\*|\*|__)", "", text)

    # Remove headers
    text = re.sub(r"^\s*#{1,6}\s*", "", text, flags=re.MULTILINE)

    # Remove blockquotes
    text = re.sub(r"^\s*>\s*", "", text, flags=re.MULTILINE)

    # Remove markdown links but keep text
    text = re.sub(r"\[([^\]]+)\]\([^)]+\)", r"\1", text)

    # Remove unordered list markers
    text = re.sub(r"^\s*[-+*]\s+", "", text, flags=re.MULTILINE)

    # Remove ordered list markers
    text = re.sub(r"^\s*\d+\.\s+", "", text, flags=re.MULTILINE)

    # Normalize spacing
    text = re.sub(r"\n{3,}", "\n\n", text).strip()

    return text
code_language = 0

In [24]:
def ask_qwen(prompt):
    messages = [
        {"role": "system", "content": "You are SV2 Analyzer, Digital Verification Assistant for analyzing and revising Testbenches. You must be proficient in Verilog, System Verilog and UVM"},
        {"role": "user", "content": f'{prompt} give the answer without giving long code snippets'}
    ]
    text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )
    model_inputs = tokenizer([text], return_tensors="pt").to(model.device)
    
    generated_ids = model.generate(
        **model_inputs,
        max_new_tokens=700,
        top_p=0.9,
        do_sample = True,
        temperature=0.85
    )
    generated_ids = [
        output_ids[len(input_ids):] for input_ids, output_ids in zip(model_inputs.input_ids, generated_ids)
    ]
    
    
    response = tokenizer.batch_decode(generated_ids, skip_special_tokens=True)[0]
    resp = markdown_to_plain_text(response) 
    return resp

In [25]:
def interpret_code (query , code_language):
    if(code_language == 0 or code_language == 1): #verilog code / systemverilog
        retrieved = vectordb[code_language].similarity_search(query, k=3)
    else: #uvm
        retrieved = vectordb[code_language].similarity_search(query, k=3)
        retrieved.extend(vectordb[code_language-1].similarity_search(query, k=1)) #getting info from the system verilog std
    answer = "\n\n".join([ret.page_content for ret in retrieved])
    meta_data = "\n".join(
        f"Page {ret.metadata.get('page_label')} — Source: {Path(ret.metadata.get('source', 'unknown')).name}"
        for ret in retrieved
    )
    interpretation = ask_qwen(f'Explain this code functionality to make it more understoodable: {query} with taking into consideration this text from the IEEE relevant standard {answer}.')
    return f'{interpretation}\n{meta_data}'




### Suggesting Enhancements for the code architecture

In [26]:
def suggest_enhancement (query , code_language):
    if(code_language == 0 or code_language == 1): #verilog code / systemverilog
        retrieved = vectordb[code_language].similarity_search(query, k=3)
    else: #uvm
        retrieved = vectordb[code_language].similarity_search(query, k=3)
        retrieved.extend(vectordb[code_language-1].similarity_search(query, k=1)) #getting info from the system verilog std
    answer = "\n\n".join([ret.page_content for ret in retrieved])
    meta_data = "\n".join(
        f"Page {ret.metadata.get('page_label')} — Source: {Path(ret.metadata.get('source', 'unknown')).name}"
        for ret in retrieved
    )
    suggestion= ask_qwen(f'Suggest enhancements for this code to be more professional and achieve more signal coverage: {query} with taking into consideration this text from the IEEE relevant standard {answer}.')
    return f'{suggestion}\n{meta_data}'




### Assuring the code (syntax and logic) wise

In [27]:
def check_code (query , code_language):
    if(code_language == 0 or code_language == 1): #verilog code / systemverilog
        retrieved = vectordb[code_language].similarity_search(query, k=3)
    else: #uvm
        retrieved = vectordb[code_language].similarity_search(query, k=3)
        retrieved.extend(vectordb[code_language-1].similarity_search(query, k=1)) #getting info from the system verilog std
    answer = "\n\n".join([ret.page_content for ret in retrieved])
    meta_data = "\n".join(
        f"Page {ret.metadata.get('page_label')} — Source: {Path(ret.metadata.get('source', 'unknown')).name}"
        for ret in retrieved
    )
    check_result = ask_qwen(f'How syntically and logicallt this code is correct: {query} with taking into consideration this text from the IEEE relevant standard {answer}.')
    return f'{check_result}\n{meta_data}'




### General Query

In [28]:
def ask_qwen_rag (query , code_language):
    if(code_language == 0 or code_language == 1): #verilog code / systemverilog
        retrieved = vectordb[code_language].similarity_search(query, k=3)
    else: #uvm
        retrieved = vectordb[code_language].similarity_search(query, k=3)
        retrieved.extend(vectordb[code_language-1].similarity_search(query, k=1)) #getting info from the system verilog std
    answer = "\n\n".join([ret.page_content for ret in retrieved])
    meta_data = "\n".join(
        f"Page {ret.metadata.get('page_label')} — Source: {Path(ret.metadata.get('source', 'unknown')).name}"
        for ret in retrieved
    )
    check_result = ask_qwen(f'{query} with taking into consideration this text from the IEEE relevant standard {answer}.')
    return f'{check_result}\n{meta_data}'




### Summary

In [29]:
def sum_resp(last_response):
    messages=[
        { 'role' : 'system' , 'content' : "you are proficient digital verification analyzer assistant. Your duty is to summarize the given text only. Ignore any provided resources."},
        { 'role': 'user', 'content': f' Do not reply with any additional text just summarize and revise the following {last_response}'}
    ]
    inputs = tokenizer_summary.apply_chat_template(messages, add_generation_prompt=True, return_tensors="pt").to(model_summary.device)
    # tokenizer.eos_token_id is the id of <|EOT|> token
    outputs = model_summary.generate(inputs, max_new_tokens=700, do_sample=True, num_return_sequences=1, eos_token_id=tokenizer_summary.eos_token_id)
    resp = tokenizer_summary.decode(outputs[0][len(inputs[0]):], skip_special_tokens=True)
    return resp

In [37]:
torch.cuda.empty_cache()
import gc
gc.collect()

0

# NGROK Part

In [31]:
!pip install fastapi uvicorn pyngrok

huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


In [32]:
NGROK_TOKEN = "387hElQjmw3doRDJ5WThTpOiKPu_4imXopRVhsLqjFF4q2UvM"
API_KEY = "secretSV_A123"

MODE_FUNCTIONS = {
    "Interpretation": 0,
    "Inspection": 1,
    "Parsing": 2,
    "Consultation": 3,
    "Ask": 4
}

Languages = {
    "Verilog": 0,
    "SystemVerilog": 1,
    "UVM": 2,
    }



In [33]:
from fastapi import FastAPI, Request, HTTPException
import uvicorn, threading, time, socket
from pyngrok import ngrok, conf

assistant_response = ""

app = FastAPI()

@app.post("/sv2analyzer")
async def sv2analyzer(req: Request):
    global assistant_response 
    if req.headers.get("authorization") != f"Bearer {API_KEY}":
        raise HTTPException(status_code=401, detail="Unauthorized")
    
    data = await req.json()  # dict

    # extract each field individually
    prompt = data.get("prompt", "")
    mode = data.get("mode", "Ask")
    code_lan = data.get("code_lan", "1")
    summarize = data.get("summarize_state" , 0)

    if mode not in MODE_FUNCTIONS:
        raise HTTPException(status_code=400, detail=f"Unknown mode '{mode}'")
    if(summarize == 1 ):
        assistant_response = sum_resp(prompt)
    else:
        if(MODE_FUNCTIONS[mode]==0):
           assistant_response = interpret_code(prompt , Languages[code_lan])
        elif(MODE_FUNCTIONS[mode]==1):
            assistant_response = check_code(prompt , Languages[code_lan])
        elif(MODE_FUNCTIONS[mode]==2):
            assistant_response = parse_query(prompt , format_instructions)
        elif(MODE_FUNCTIONS[mode]==3):
            assistant_response = suggest_enhancement(prompt , Languages[code_lan])
        elif(MODE_FUNCTIONS[mode]==4):
            assistant_response = ask_qwen_rag(prompt , Languages[code_lan])
        
    
    return {"response": assistant_response}

    


In [34]:
def free_port():
    s = socket.socket()
    s.bind(('', 0))
    port = s.getsockname()[1]
    s.close()
    return port

port = free_port()
conf.get_default().auth_token = NGROK_TOKEN
public_url = ngrok.connect(port).public_url
print("Your public URL:", public_url)

def run(): uvicorn.run(app, host="0.0.0.0", port=port)
threading.Thread(target=run, daemon=True).start()
time.sleep(1)

Your public URL: https://osculant-danna-inclinational.ngrok-free.dev                                


INFO:     Started server process [55]
INFO:     Waiting for application startup.
INFO:     Application startup complete.
INFO:     Uvicorn running on http://0.0.0.0:47433 (Press CTRL+C to quit)
The attention mask and the pad token id were not set. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
Setting `pad_token_id` to `eos_token_id`:32021 for open-end generation.
The attention mask is not set and cannot be inferred from input because pad token is same as eos token. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.


INFO:     154.179.19.112:0 - "POST /sv2analyzer HTTP/1.1" 200 OK
